In [1]:
import torch
import pandas as pd
from tqdm import tqdm
import numpy as np

from src.models.optimized.original import Ex2VecOriginalFast

In [2]:
df = pd.read_parquet('../../../../sorted_data.parquet')


In [3]:
max_history = 3500
user_count = df['user_id'].max()
item_count = df['track_id'].max()

In [ ]:
time_history = np.zeros((user_count+1, max_history), dtype=int)
item_history = np.zeros((user_count+1, max_history), dtype=int)

for user in tqdm(df['user_id'].unique()):
    tmp = df[df['user_id'] == user]
    item_history[user, :len(tmp)] = tmp['track_id'].to_numpy()
    time_history[user, :len(tmp)] = tmp['ts'].to_numpy()

 64%|██████▍   | 8427/13209 [05:31<05:32, 14.39it/s]

In [ ]:
# lets remove some from the history based on our test thingies
import json

with open('../../../../split_data/test/test_dict.json', 'r') as f:
    data = json.load(f)





In [ ]:
for user, items in data.items():
    np.put(item_history[int(user), :], items, [0,0])

In [ ]:
from torch.utils.data import Dataset

class Ex2VecDataset(Dataset):
    def __init__(self, times, users, items):
        self.times = times
        self.users = users
        self.items = items

    def __len__(self):
        return len(self.times)

    def __getitem__(self, idx):
        return self.times[idx], self.users[idx], self.items[idx]

In [ ]:
import torch
from torch.utils.data import DataLoader

data = Ex2VecDataset(df['ts'].to_numpy(), df['user_id'].to_numpy(), df['track_id'].to_numpy())

batch_size = 128

# train_dataloader = DataLoader(data, batch_size=128, shuffle=True)

In [ ]:
device = 'cuda'
config = {'n_users': user_count, 'n_items': item_count, 'latent_d': 64}

model = Ex2VecOriginalFast(config)
model.initialize_histories(torch.tensor(item_history).to(device), torch.tensor(time_history).to(device))

In [10]:
model.to(device)

log_every = 100

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

use_amp = bool(device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

global_step = 0
history = {"train_loss": [], "val_loss": []}

for epoch in range(100):
    model.train()
    epoch_loss = 0.0
    n = 0
    t0 = time.time()
    train_dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)
    for i, batch in enumerate(train_dataloader):
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(x)
            loss = criterion(outputs, y)

        # Backprop (AMP-aware)
        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        bs = y.shape[0] if torch.is_tensor(y) and y.ndim > 0 else 1
        epoch_loss += float(loss.item()) * bs
        n += bs
        global_step += 1

        if (step % log_every == 0):
            lr = optimizer.param_groups[0]["lr"]
            avg_loss = epoch_loss / max(n, 1)
            elapsed = time.time() - t0
            print(
                f"[epoch {epoch}/100] "
                f"step {step}/{len(train_dataloader)} "
                f"loss={avg_loss:.4f} lr={lr:.2e} "
                f"time={elapsed:.1f}s"
            )
    train_loss = epoch_loss / max(n, 1)
    history["train_loss"].append(train_loss)

    # Scheduler step (epoch-based)
    scheduler.step()

ckpt = {
    "model_state_dict": model.state_dict(),
    "config": vars(config),
}
ckpt["optimizer_state_dict"] = optimizer.state_dict()

torch.save(ckpt, './original.pt')
print(f"Saved final checkpoint to: original.pt")

return {"history": history, "save_path": config.save_path}


In [11]:
output = model.forward(batch[0].to(device), batch[1].to(device))

In [15]:
config

{'n_users': np.int64(13209), 'n_items': np.int64(3027), 'latent_d': 64}

In [17]:
loss = torch.nn.CrossEntropyLoss()

In [19]:
loss(output, batch[2].to(device))

tensor(13.1227, device='cuda:0', grad_fn=<NllLossBackward0>)